In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# CPC main-group sensitivity

Calculate the SI S3.6 comparison by keeping the matched patent pairs fixed, collapsing CPC symbols to the whitespace-free text before `/`, and recomputing extra codes and GBBs.

Read the main-group model trained by [dataprep 02](../dataprep/02_cpc_corpus_preparation.ipynb), the detailed projection and PATSTAT `tls225` through the paths in the first cell. Save main-group results under `DATA/main_groups/` and `OUTPUTS/main_groups/`.


In [ ]:
import numpy as np
import pandas as pd
from embedding_utils import write_projection
from gbb_utils import (ordinary_codes, read_profile, nonnoise, extra_codes,
    code_profile, reduce_profile, cluster_profile, hierarchy_correspondence,
    correspondence_metrics, membership_metrics)


## Select matched pairs and project main-group codes

Use pairs from the detailed projection that have passed the text-distance and common-subclass filters and have main-group codes in both families. Normalize vectors for pair similarity; use raw model vectors for clustering.


In [ ]:
from gensim.models import Word2Vec

work = DATA / "main_groups"
out = OUTPUTS / "main_groups"
work.mkdir(parents=True, exist_ok=True)
out.mkdir(parents=True, exist_ok=True)
vectors = Word2Vec.load(str(BASELINE / "external/cpc8d_noy02_sg_ns_dim50.model")).wv
unit_vectors = dict(zip(vectors.index_to_key, vectors.get_normed_vectors()))
original = pd.read_parquet(BASELINE / "green_proj_cosangle.parquet")
pairs = original[["ccmt_patent_id", "patent_id", "cos"]].drop_duplicates()
pairs = pairs.sort_values(["ccmt_patent_id", "cos", "patent_id"]).reset_index(drop=True)
pairs["rank"] = pairs.groupby("ccmt_patent_id").cumcount() + 1
cpc = pd.read_parquet(PATSTAT / "tls225.parquet", columns=["docdb_family_id", "cpc_class_symbol"])
cpc.columns = ["patent_id", "cpc"]
cpc = cpc.dropna(subset=["cpc"])
cpc["cpc"] = ordinary_codes(cpc.cpc).str.split("/").str[0]
families = pd.unique(pairs[["ccmt_patent_id", "patent_id"]].to_numpy().ravel())
cpc = cpc.loc[cpc.patent_id.isin(families) & cpc.cpc.isin(vectors.key_to_index)
              & ~cpc.cpc.str.startswith("Y")].drop_duplicates()
available = cpc.patent_id.unique()
pairs = pairs.loc[pairs.ccmt_patent_id.isin(available) & pairs.patent_id.isin(available)]
pairs.to_parquet(work / "retained_pairs.parquet", index=False)
# Use the matched pairs selected by the detailed projection.
write_projection(pairs, cpc, unit_vectors, work / "projection.parquet")


## Identify extra codes on the same matched pairs

In [ ]:
coarse = pd.read_parquet(work / "projection.parquet")
original = original.merge(pairs[["ccmt_patent_id", "patent_id", "rank"]], on=["ccmt_patent_id", "patent_id"])
detailed_members = extra_codes(original)
detailed_members["cpc"] = ordinary_codes(detailed_members.cpc).str.split("/").str[0]
detailed_members = detailed_members.loc[detailed_members.cpc.isin(vectors.key_to_index), ["ccmt_patent_id", "cpc"]].drop_duplicates()
coarse_members = extra_codes(coarse)


## Cluster the coarse profile and compare through the CPC hierarchy

Scale the minimum cluster size by the number of codes in the coarse versus detailed profiles. Keep weights for codes without a corresponding cluster in the saved table. Calculate the cluster-comparison statistic using codes with a corresponding cluster.

In [ ]:
reference = read_profile(BASELINE / "clusangle_outlier_hdbscan2.parquet")
profile = code_profile(coarse_members)
min_size = max(2, int(np.floor(100 * len(profile) / len(reference) + .5)))
coordinates = reduce_profile(profile, vectors, seed=SEED)
partition = cluster_profile(profile, coordinates, min_size=min_size, prefix="m")
partition.to_parquet(out / "gbbs.parquet", index=False)
cells = hierarchy_correspondence(reference, partition)
cells.to_csv(out / "hierarchy_correspondence.csv", index=False)
# Unmapped mass is retained in the saved table; block correspondence uses mapped mass.
mapped = cells.loc[cells.clus_reference != "__unmapped__"]
summary = dict(clusters=nonnoise(partition).clus.nunique(), min_cluster_size=min_size,
               **correspondence_metrics(mapped),
               frequency_cosine=membership_metrics(detailed_members, coarse_members)["frequency_cosine"])
pd.DataFrame([summary]).to_csv(out / "comparison.csv", index=False)
rows = []
for name, members in [("detailed_collapsed", detailed_members), ("main_group", coarse_members)]:
    members.to_parquet(out / f"{name}_memberships.parquet", index=False)
    rows.append(dict(representation=name, retained_focals=pairs.ccmt_patent_id.nunique(),
                     positive_focals=members.ccmt_patent_id.nunique(), memberships=len(members)))
pd.DataFrame(rows).to_csv(out / "abundance.csv", index=False)
print(summary)
